# Notebook 02_99 — Merge Feature Selection Results

Combines the 6 per-method CSVs (`results/fs_*_raw.csv`) produced by notebooks
`02_01` .. `02_06`, then produces:
- Mean ± std summary table
- Best FS method per classifier
- F1 heatmaps (method × K, per classifier)
- Wilcoxon significance test vs the notebook-01 baseline

In [ ]:
import sys; sys.path.insert(0, '/home/anhvan/Hanh/notebook')
from common import *
from scipy.stats import wilcoxon
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns

print('Imports OK')

## 1. Load and Concatenate All FS Results

In [ ]:
d = load_data()
CLASSIFIER_NAMES = d['classifier_names']

FS_FILES = {
    'Correlation'   : 'fs_correlation_raw.csv',
    'ChiSquare'     : 'fs_chisquare_raw.csv',
    'XGB_Importance': 'fs_xgb_importance_raw.csv',
    'SHAP'          : 'fs_shap_raw.csv',
    'RFE'           : 'fs_rfe_raw.csv',
    'Consensus'     : 'fs_consensus_raw.csv',
}

dfs = []
for method, fname in FS_FILES.items():
    path = f'{RESULTS_DIR}{fname}'
    if not os.path.exists(path):
        print(f'MISSING: {path}  -> run the corresponding 02_0X notebook first.')
        continue
    df = pd.read_csv(path)
    expected = len(K_VALUES) * len(SEEDS) * len(CLASSIFIER_NAMES)
    status = 'COMPLETE' if len(df) == expected else f'PARTIAL ({len(df)}/{expected})'
    print(f'{method:<16} {fname:<28} rows={len(df):<5} {status}')
    dfs.append(df)

fs_raw = pd.concat(dfs, ignore_index=True)
print(f'\nTotal combined rows: {len(fs_raw)}')

## 2. Mean ± Std Summary Table

In [ ]:
rows = []
for method in fs_raw['method'].unique():
    for K in sorted(fs_raw['K'].unique()):
        for clf in CLASSIFIER_NAMES:
            sub = fs_raw[(fs_raw['method'] == method) & (fs_raw['K'] == K) & (fs_raw['classifier'] == clf)]
            if sub.empty:
                continue
            row = {'FS_Method': method, 'K': K, 'Classifier': clf}
            for m in METRICS:
                row[m] = f"{sub[m].mean():.4f} ± {sub[m].std(ddof=1):.4f}"
            rows.append(row)

fs_summary = pd.DataFrame(rows)
fs_summary.to_csv(f'{RESULTS_DIR}fs_results_summary.csv', index=False)
print(fs_summary.head(30).to_string(index=False))
print(f'\nSaved: {RESULTS_DIR}fs_results_summary.csv')

## 3. Best FS Method Per Classifier

In [ ]:
print('=== BEST FS METHOD PER CLASSIFIER (highest mean F1) ===')
best_rows = []
for clf in CLASSIFIER_NAMES:
    sub = fs_raw[fs_raw['classifier'] == clf]
    grouped = sub.groupby(['method', 'K'])['f1'].mean().reset_index()
    if grouped.empty:
        continue
    best = grouped.loc[grouped['f1'].idxmax()]
    train_t = sub[(sub['method'] == best['method']) & (sub['K'] == best['K'])]['train_time_s'].mean()
    print(f"  {clf:6s}  best FS={best['method']:<16} K={int(best['K'])}  F1={best['f1']:.4f}  train={train_t:.1f}s")
    best_rows.append({'Classifier': clf, 'Best_FS': best['method'], 'K': int(best['K']),
                       'F1': best['f1'], 'train_time_s': train_t})

pd.DataFrame(best_rows).to_csv(f'{RESULTS_DIR}fs_best_per_classifier.csv', index=False)

## 4. Heatmaps — F1 by Method × K (per classifier)

In [ ]:
FS_METHODS = list(FS_FILES.keys())
K_present  = sorted(fs_raw['K'].unique())

n_clf = len(CLASSIFIER_NAMES)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for ax, clf in zip(axes, CLASSIFIER_NAMES):
    matrix = np.full((len(FS_METHODS), len(K_present)), np.nan)
    for i, fs in enumerate(FS_METHODS):
        for j, K in enumerate(K_present):
            sub = fs_raw[(fs_raw['method'] == fs) & (fs_raw['K'] == K) & (fs_raw['classifier'] == clf)]
            if not sub.empty:
                matrix[i, j] = sub['f1'].mean()
    sns.heatmap(matrix, annot=True, fmt='.3f', cmap='YlOrRd',
                xticklabels=[f'K={k}' for k in K_present],
                yticklabels=FS_METHODS, ax=ax, vmin=0, vmax=1)
    ax.set_title(f'{clf} — Macro F1')

for ax in axes[n_clf:]:
    ax.set_visible(False)

plt.suptitle('Feature Selection: Macro F1 by Method × K (mean over completed seeds)', y=1.02)
plt.tight_layout()
plt.savefig(f'{RESULTS_DIR}fs_f1_heatmap.png', dpi=100, bbox_inches='tight')
plt.close()
print(f'Saved: {RESULTS_DIR}fs_f1_heatmap.png')

## 5. Statistical Significance vs Baseline (Wilcoxon)

In [ ]:
baseline_path = f'{RESULTS_DIR}baseline_raw_scores.json'
if not os.path.exists(baseline_path):
    print('Baseline scores not found — run 01_baseline.ipynb first.')
else:
    with open(baseline_path) as f:
        baseline_scores = json.load(f)

    sig_rows = []
    for clf in CLASSIFIER_NAMES:
        base_f1 = baseline_scores[clf]['f1']
        for fs in FS_METHODS:
            for K in K_present:
                sub = fs_raw[(fs_raw['method'] == fs) & (fs_raw['K'] == K) & (fs_raw['classifier'] == clf)]
                if len(sub) != len(SEEDS):
                    continue  # only test fully-completed combinations
                fs_f1 = sub.sort_values('seed')['f1'].tolist()
                try:
                    _, p = wilcoxon(fs_f1, base_f1, alternative='two-sided')
                except Exception:
                    p = float('nan')
                sig_rows.append({
                    'Classifier': clf, 'FS': fs, 'K': K,
                    'FS_F1_mean': np.mean(fs_f1), 'Baseline_F1_mean': np.mean(base_f1),
                    'p_value': p, 'significant (alpha=0.05)': 'Yes' if p < 0.05 else 'No'
                })

    sig_df = pd.DataFrame(sig_rows)
    sig_df.to_csv(f'{RESULTS_DIR}fs_significance.csv', index=False)
    print(f'Saved: {RESULTS_DIR}fs_significance.csv  ({len(sig_df)} rows)')
    print(sig_df.head(20).to_string(index=False))

## 6. Completion Check

In [ ]:
expected_total = len(FS_FILES) * len(K_VALUES) * len(SEEDS) * len(CLASSIFIER_NAMES)
print(f'Expected total rows : {expected_total}')
print(f'Actual total rows   : {len(fs_raw)}')
if len(fs_raw) < expected_total:
    print('\n⚠ Some experiments are missing. Re-run the corresponding 02_0X notebook(s) — they will resume automatically.')
else:
    print('\n✓ All FS experiments complete.')